# Food Classifier — YOLOv8 Training Pipeline

Pipeline completo: geração de dataset, augmentation, treinamento e avaliação.

In [ ]:
import os
import random
import shutil
from pathlib import Path

import cv2
import numpy as np
from tqdm import tqdm
from ultralytics import YOLO

DATASET_BASE_DIR = Path('dataset_base')
DATASET_DIR = Path('dataset')
DATA_YAML = Path('data.yaml')
RUNS_DIR = Path('runs')

YOLO_CLASSES = ['arroz', 'feijao', 'acucar', 'cafe', 'macarrao']
CLASS_MAP = {cls: i for i, cls in enumerate(YOLO_CLASSES)}

IMG_SIZE = 640
IMAGES_PER_INPUT = 10
SPLIT_RATIO = 0.2
SEED = 42
YOLO_BASE_MODEL = 'yolov8n.pt'
EPOCHS = 30
BATCH_SIZE = 8
TRAIN_NAME = 'treino_alimentos'
SUPPORTED_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

print('Configuração carregada.')

In [ ]:
def random_rotate(img):
    if random.random() < 0.7:
        angle = random.uniform(-20, 20)
        h, w = img.shape[:2]
        M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
        return cv2.warpAffine(img, M, (w, h))
    return img

def random_flip(img):
    if random.random() < 0.5:
        return cv2.flip(img, random.choice([-1, 0, 1]))
    return img

def adjust_brightness_contrast(img):
    if random.random() < 0.5:
        alpha = random.uniform(0.8, 1.2)
        beta = random.randint(-30, 30)
        return np.clip(img.astype(np.float32) * alpha + beta, 0, 255).astype(np.uint8)
    return img

def random_blur(img):
    if random.random() < 0.3:
        k = random.choice([3, 5])
        return cv2.GaussianBlur(img, (k, k), 0)
    return img

def random_noise(img):
    if random.random() < 0.3:
        noise = np.random.normal(0, 10, img.shape).astype(np.int16)
        return np.clip(img.astype(np.int16) + noise, 0, 255).astype(np.uint8)
    return img

def random_zoom(img):
    if random.random() < 0.5:
        scale = random.uniform(0.85, 1.15)
        h, w = img.shape[:2]
        nh, nw = int(h * scale), int(w * scale)
        resized = cv2.resize(img, (nw, nh))
        if scale > 1:
            sy = (nh - h) // 2
            sx = (nw - w) // 2
            return resized[sy:sy + h, sx:sx + w]
        pad_y = (h - nh) // 2
        pad_x = (w - nw) // 2
        out = np.zeros_like(img)
        out[pad_y:pad_y + nh, pad_x:pad_x + nw] = resized
        return out
    return img

def augment_image(img):
    img = random_rotate(img)
    img = random_flip(img)
    img = adjust_brightness_contrast(img)
    img = random_blur(img)
    img = random_noise(img)
    img = random_zoom(img)
    return cv2.resize(img, (IMG_SIZE, IMG_SIZE))

print('Funções de augmentation definidas.')

In [ ]:
for category in YOLO_CLASSES:
    out_dir = DATASET_DIR / 'images' / 'train' / category
    out_dir.mkdir(parents=True, exist_ok=True)

    src = DATASET_BASE_DIR / category
    if not src.exists():
        print(f'[AVISO] {src} nao encontrado, pulando.')
        continue

    images = [p for p in src.iterdir() if p.suffix.lower() in SUPPORTED_EXTENSIONS]
    for img_path in tqdm(images, desc=f'Augmenting {category}'):
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        base = out_dir / img_path.stem
        cv2.imwrite(str(base) + '_orig.jpg', img)
        for i in range(IMAGES_PER_INPUT):
            cv2.imwrite(str(base) + f'_aug_{i}.jpg', augment_image(img))

print('Dataset gerado.')

In [ ]:
random.seed(SEED)
for category in YOLO_CLASSES:
    src = DATASET_DIR / 'images' / 'train' / category
    dst = DATASET_DIR / 'images' / 'val' / category
    dst.mkdir(parents=True, exist_ok=True)
    if not src.exists():
        continue
    imgs = [p for p in src.iterdir() if p.suffix.lower() in SUPPORTED_EXTENSIONS]
    random.shuffle(imgs)
    val_imgs = imgs[:int(len(imgs) * SPLIT_RATIO)]
    for p in val_imgs:
        shutil.move(str(p), str(dst / p.name))

print('Split train/val concluído.')

In [ ]:
for split in ['train', 'val']:
    for category in YOLO_CLASSES:
        img_dir = DATASET_DIR / 'images' / split / category
        lbl_dir = DATASET_DIR / 'labels' / split / category
        lbl_dir.mkdir(parents=True, exist_ok=True)
        if not img_dir.exists():
            continue
        class_id = CLASS_MAP[category]
        for img_path in img_dir.iterdir():
            if img_path.suffix.lower() not in SUPPORTED_EXTENSIONS:
                continue
            (lbl_dir / (img_path.stem + '.txt')).write_text(f'{class_id} 0.5 0.5 0.8 0.8\n')

DATA_YAML.write_text(
    f'path: {DATASET_DIR.resolve()}\ntrain: images/train\nval: images/val\n\nnames:\n'
    + ''.join(f'  {i}: {cls}\n' for i, cls in enumerate(YOLO_CLASSES))
)

print('Labels e data.yaml gerados.')

In [ ]:
model = YOLO(YOLO_BASE_MODEL)
results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    name=TRAIN_NAME,
    project=str(RUNS_DIR / 'detect'),
)
print('Treinamento concluído.')

In [ ]:
weights_path = RUNS_DIR / 'detect' / TRAIN_NAME / 'weights' / 'best.pt'
eval_model = YOLO(str(weights_path))
metrics = eval_model.val(data=str(DATA_YAML), imgsz=IMG_SIZE)

print(f'mAP50:    {metrics.box.map50:.4f}')
print(f'mAP50-95: {metrics.box.map:.4f}')
for i, name in enumerate(YOLO_CLASSES):
    print(f'  {name}: P={metrics.box.p[i]:.3f}  R={metrics.box.r[i]:.3f}  AP50={metrics.box.ap50[i]:.3f}')

In [ ]:
import matplotlib.pyplot as plt

sample_images = []
for category in YOLO_CLASSES:
    imgs = list((DATASET_DIR / 'images' / 'val' / category).glob('*.jpg'))
    if imgs:
        sample_images.append(imgs[0])
    if len(sample_images) >= 3:
        break

for img_path in sample_images:
    res = eval_model(str(img_path), conf=0.5, verbose=False)
    annotated = res[0].plot()
    plt.figure(figsize=(6, 6))
    plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    plt.title(img_path.parent.name)
    plt.axis('off')
    plt.show()